In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import WeightedRandomSampler
from torch_geometric.nn.conv import GATConv, GCNConv, SplineConv
from torch_geometric.nn.norm import GraphNorm
from torch_geometric.nn import Sequential
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
import torch.utils.data as torchdata
import tqdm
from math import floor
import gc
import matplotlib.pyplot as plt
from Model.MENDR.MENDR import MENDR_model
from Model.MENDR.MENDREncoder import MENDRPatchEncoder
from Model.MENDR.mAtt.optimizer import MixOptimizer
from Model.MENDR.mAtt.spd import SPDTangentSpace
import random
import os
import math
from Datasets.datasetTUEV import WaveletTUEVDataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, precision_recall_curve, auc, f1_score
from torchjd import mtl_backward
from torchjd.aggregation import UPGrad

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
print("*" * 50)
finetune_train_dataset = WaveletTUEVDataset(root="/home/hice1/mchen439/scratch/TUEV/train", frac=1.0, split="train")
print("Dataset Loaded. Length of Train Dataset: ", len(finetune_train_dataset))
finetune_eval_dataset = WaveletTUEVDataset(root="/home/hice1/mchen439/scratch/TUEV/eval", frac=1.0, split="eval")
print("Dataset Loaded. Length of Validation Dataset: ", len(finetune_eval_dataset))

**************************************************
Loading 83932 folders


 15%|█▍        | 12284/83932 [00:33<02:59, 398.05it/s]

In [ ]:
print(finetune_eval_dataset[0]['graph'])
print(finetune_eval_dataset[0]['delta'].shape)
print(finetune_eval_dataset[0]['theta'].shape)
print(finetune_eval_dataset[0]['alpha'].shape)
print(finetune_eval_dataset[0]['beta'].shape)
print(finetune_eval_dataset[0]['gamma'].shape)
print(finetune_eval_dataset[0]['graph'].y)
print(finetune_eval_dataset[1]['graph'].y)
print(finetune_eval_dataset[2]['graph'].y)
print(finetune_eval_dataset[3]['graph'].y)
print(finetune_eval_dataset[4]['graph'].y)
print(finetune_eval_dataset[5]['graph'].y)
print(finetune_eval_dataset[6]['graph'].y)
print(finetune_eval_dataset[7]['graph'].y)
print(finetune_eval_dataset[8]['graph'].y)
print(finetune_eval_dataset[9]['graph'].y)
print(finetune_eval_dataset[29000]['graph'].y)
print(finetune_eval_dataset[29010]['graph'].y)
print(finetune_eval_dataset[29020]['graph'].y)
print(finetune_eval_dataset[29030]['graph'].y)

Data(edge_index=[2, 361], edge_attr=[361, 1], pos=[19, 3], y=[1])
torch.Size([19, 86])
torch.Size([19, 86])
torch.Size([19, 166])
torch.Size([19, 326])
torch.Size([19, 645])
tensor([5])
tensor([5])
tensor([4])
tensor([5])
tensor([1])
tensor([5])
tensor([5])
tensor([4])
tensor([0])
tensor([5])
tensor([5])
tensor([5])
tensor([1])
tensor([5])


In [ ]:
train_label_count = {
    0: 0,
    1: 0,
    2: 0,
    3: 0,
    4: 0,
    5: 0
}
y_train = []
for i in range(len(finetune_train_dataset)):
    graph = finetune_train_dataset[i]['graph']
    train_label = graph.y.item()
    assert train_label >= 0 and train_label <= 5, f"Label {train_label} is not in range [0, 5]"
    train_label_count[train_label] += 1
    y_train.append(train_label)

print(train_label_count)

{0: 645, 1: 11254, 2: 6184, 3: 1070, 4: 11053, 5: 53726}


In [ ]:
test_label_count = {
    0: 0,
    1: 0,
    2: 0,
    3: 0,
    4: 0,
    5: 0
}

y_test = []
for i in range(len(finetune_eval_dataset)):
    graph = finetune_eval_dataset[i]['graph']
    test_label = graph.y.item()
    assert test_label >= 0 and test_label <= 5, f"{test_label} is not a valid label"
    test_label_count[test_label] += 1
    y_test.append(test_label)

print(test_label_count)

{0: 567, 1: 4677, 2: 1998, 3: 329, 4: 2204, 5: 19646}


In [ ]:
train_label_counts_arr = np.array(list(train_label_count.values()))
train_sampler_class_weight = 1. / torch.from_numpy(train_label_counts_arr)
train_samples_weight = np.array([train_sampler_class_weight[t] for t in y_train])
train_sampler = WeightedRandomSampler(weights=train_samples_weight, num_samples=len(train_samples_weight))

test_label_counts_arr = np.array(list(test_label_count.values()))
test_sampler_class_weight = 1. / torch.from_numpy(test_label_counts_arr)
test_samples_weight = np.array([test_sampler_class_weight[t] for t in y_test])
test_sampler = WeightedRandomSampler(weights=test_samples_weight, num_samples=len(test_samples_weight))

print(train_sampler_class_weight)
print(test_sampler_class_weight)
print(train_samples_weight.shape)
print(test_samples_weight.shape)

tensor([1.5504e-03, 8.8857e-05, 1.6171e-04, 9.3458e-04, 9.0473e-05, 1.8613e-05])
tensor([1.7637e-03, 2.1381e-04, 5.0050e-04, 3.0395e-03, 4.5372e-04, 5.0901e-05])
(83932,)
(29421,)


In [8]:
finetune_train_loader = DataLoader(finetune_train_dataset, batch_size=256, num_workers=8, pin_memory=True, persistent_workers=True, shuffle=True)
finetune_eval_loader = DataLoader(finetune_eval_dataset, batch_size=256, num_workers=8, pin_memory=True, persistent_workers=True, shuffle=False)
print(len(finetune_train_loader), len(finetune_eval_loader))
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']

656 230


In [9]:
class FinetuneDecoder(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.tangent = SPDTangentSpace(19, device=device)
        self.flatten = nn.Flatten()

        self.band_tangent_spaces = nn.ParameterDict({
            'delta': SPDTangentSpace(19, device=device),
            'theta': SPDTangentSpace(19, device=device),
            'alpha': SPDTangentSpace(19, device=device),
            'beta':  SPDTangentSpace(19, device=device),
            'gamma': SPDTangentSpace(19, device=device),
        })

        self.band_flatten = nn.ParameterDict({
            'delta': nn.Flatten(),
            'theta': nn.Flatten(),
            'alpha': nn.Flatten(),
            'beta':  nn.Flatten(),
            'gamma': nn.Flatten(),
        })

        self.band_dim = {
            'delta': 19,
            'theta': 19,
            'alpha': 19,
            'beta':  19,
            'gamma': 19,
        }

        self.flattened = (19 * 20) // 2

        self.band_decoders = nn.ParameterDict({
            'delta': nn.Sequential(nn.GELU(), nn.LayerNorm(self.flattened), nn.Dropout(p=0.2), nn.Linear(self.flattened, 10), nn.LayerNorm(10), nn.Dropout(p=0.1)),
            'theta': nn.Sequential(nn.GELU(), nn.LayerNorm(self.flattened), nn.Dropout(p=0.2), nn.Linear(self.flattened, 10), nn.LayerNorm(10), nn.Dropout(p=0.1)),
            'alpha': nn.Sequential(nn.GELU(), nn.LayerNorm(self.flattened), nn.Dropout(p=0.2), nn.Linear(self.flattened, 10), nn.LayerNorm(10), nn.Dropout(p=0.1)),
            'beta':  nn.Sequential(nn.GELU(), nn.LayerNorm(self.flattened), nn.Dropout(p=0.2), nn.Linear(self.flattened, 10), nn.LayerNorm(10), nn.Dropout(p=0.1)),
            'gamma': nn.Sequential(nn.GELU(), nn.LayerNorm(self.flattened), nn.Dropout(p=0.2), nn.Linear(self.flattened, 10), nn.LayerNorm(10), nn.Dropout(p=0.1)),
        })
        self.combined_seq = nn.Sequential(nn.LayerNorm(19 * 10), nn.Dropout(p=0.2), nn.Linear(19*10, 10), nn.GELU())
        self.final_decoder = nn.Linear(5*10 + 10, 6)

    def forward(self, wavelet_manifold_output, combined_manifold_output):
        batch_size = combined_manifold_output.shape[0]
        num_patches = combined_manifold_output.shape[1]
        embedding_dim = combined_manifold_output.shape[2]
        
        band_decodings = []
        for band, encodings in wavelet_manifold_output.items():
            #print(band, encodings.shape, self.band_dim[band])
            band_tangent = self.band_tangent_spaces[band](encodings.view(batch_size*num_patches, self.band_dim[band], self.band_dim[band]))
            band_tangent = self.band_flatten[band](band_tangent)
            band_decoding = self.band_decoders[band](band_tangent)
            band_decodings.append(band_decoding)
        band_decodings = torch.cat(band_decodings, dim=1)

        x = self.tangent(combined_manifold_output.view(batch_size*num_patches, embedding_dim, embedding_dim))
        #x = x.view(batch_size, num_patches, -1)
        x = self.flatten(x)
        x = self.combined_seq(x)
        return self.final_decoder(torch.cat([x, band_decodings], dim=1))

In [10]:
model = MENDR_model(device=device, contextualizer_size="LARGE").to(device)
model.load_state_dict(torch.load("./checkpoint/4b5781b2c41e45a7a666bb8c29b4ae22_14_LARGE/mendr_model_weights.pth", weights_only=True))
model_decoder = FinetuneDecoder(device).to(device)
optim_params = []
optim_params.append({'params': model.mendr_encoder.parameters(), 'lr': 1e-5})
optim_params.append({'params': model.mendr_contextualizer.parameters(), 'lr': 1e-5})
optim_params.append({'params': model_decoder.parameters(), 'lr': 1e-5})
optimizer = MixOptimizer(torch.optim.AdamW(optim_params, lr=1e-5, weight_decay=1e-4))
optimizer.set_scheduler_t0(len(finetune_train_loader) * 15)
for p in model.mendr_contextualizer.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -1e7, 1e7))
for p in model_decoder.parameters():
    p.register_hook(lambda grad: torch.clamp(grad, -1e7, 1e7))
aggregator = UPGrad(pref_vector=torch.tensor([1, 1, 1, 1, 1, 1]).to(device))
print("Total MENDR parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total model_decoder parameters:", sum(p.numel() for p in model_decoder.parameters() if p.requires_grad))
total_encoder_params = 0
total_decoder_params = 0
for band in BANDS:
    print(f'{band} Encoder Params: {model.mendr_encoder.encoder_decoders[band].getEncoderParamCount()}')
    print(f'{band} Decoder Params: {model.mendr_encoder.encoder_decoders[band].getDecoderParamCount()}')
    total_encoder_params += model.mendr_encoder.encoder_decoders[band].getEncoderParamCount()
    total_decoder_params += model.mendr_encoder.encoder_decoders[band].getDecoderParamCount()
print(f'Wavelet Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.WaveletContextualizer.parameters() if p.requires_grad)}')
print(f'Combined Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.CombinedContextualizer.parameters() if p.requires_grad)}')
print(f'Contextualizer Params: {sum(p.numel() for p in model.mendr_contextualizer.parameters() if p.requires_grad)}')
print("Total Encoder Params: ", total_encoder_params)
print("Total Decoder Params: ", total_decoder_params)

Set Cosine Annealing with Warm Restarts optimizer T_0 to: 656
Total MENDR parameters: 2448254
Total model_decoder parameters: 14106
delta Encoder Params: 12597
delta Decoder Params: 22309
theta Encoder Params: 12597
theta Decoder Params: 22309
alpha Encoder Params: 14041
alpha Decoder Params: 101219
beta Encoder Params: 16929
beta Decoder Params: 498051
gamma Encoder Params: 22705
gamma Decoder Params: 1622675
Wavelet Contextualizer Params: 68027
Combined Contextualizer Params: 2760
Contextualizer Params: 70788
Total Encoder Params:  78869
Total Decoder Params:  2266563


In [11]:
def _fft_loss(output, target, loss_fn):
    hanning_window = torch.hann_window(output.shape[-1]).to(device)
    output_windowed = output * hanning_window.expand_as(output)
    target_windowed = target * hanning_window.expand_as(target)

    output_fft = torch.fft.fft(output_windowed, dim=-1)
    output_amplitude = torch.abs(output_fft)
    output_angle = torch.angle(output_fft)

    target_fft = torch.fft.fft(target_windowed, dim=-1)
    target_amplitude = torch.abs(target_fft)
    target_angle = torch.abs(target_fft)

    return (loss_fn(output_amplitude, target_amplitude) + (loss_fn(output_angle, target_angle)))

def calculate_band_loss(inputs, outputs, loss_fn, loss_type='real'):
    outputs_reshaped = outputs.reshape(inputs.shape[0], inputs.shape[1], inputs.shape[2], inputs.shape[3])
    if loss_type == 'real':
        loss = loss_fn(inputs, outputs_reshaped)
    elif loss_type == 'fourier':
        loss = _fft_loss(output_reshaped, inputs, loss_fn)
    else:
        loss = loss_fn(inputs, outputs_reshaped) + _fft_loss(output_reshaped, inputs, loss_fn) * 1e-2
    return loss

In [12]:
def train_one_epoch(model, epoch, decoder, optimizer, task_loss_fn, recon_loss_fn, train_loader):
	task_loss_sum = 0
	recon_loss_sum = 0
	model.eval()
	decoder.train()

	pbar = tqdm.trange(len(train_loader), desc="Training")
	data_iterator = iter(train_loader)
	balanced_accuracies = []

	truths = []
	preds = []
	for iteration in pbar:
		data = next(data_iterator)
		relevant_bands = [data['delta'].float().to(device),
							data['theta'].float().to(device),
							data['alpha'].float().to(device),
							data['beta'].float().to(device),
							data['gamma'].float().to(device)]
		inputs = dict(zip(BANDS, relevant_bands))
		patchified_inputs, encodings, decodings, wavelet_manifold_output, combined_manifold_output = model(data['graph'], inputs)
		wavelet_manifold_output_relevant_patch = dict()
		for band, wavelet_manifold in wavelet_manifold_output.items():
			wavelet_manifold_output_relevant_patch[band] = wavelet_manifold[:, 1, :, :][:, None, :, :]
		prediction = decoder(wavelet_manifold_output_relevant_patch, combined_manifold_output[:, 1, :, :][:, None, :, :]).float()
		ground_truth = data['graph'].y.to(device).long()
		task_loss = task_loss_fn(prediction, ground_truth)
		delta_loss = calculate_band_loss(patchified_inputs['delta'], decodings['delta'], recon_loss_fn)
		theta_loss = calculate_band_loss(patchified_inputs['theta'], decodings['theta'], recon_loss_fn)
		alpha_loss = calculate_band_loss(patchified_inputs['alpha'], decodings['alpha'], recon_loss_fn)
		beta_loss = calculate_band_loss(patchified_inputs['beta'],   decodings['beta'], recon_loss_fn)
		gamma_loss = calculate_band_loss(patchified_inputs['gamma'], decodings['gamma'], recon_loss_fn)
		# Borrowed from CBramod: https://github.com/wjq-learning/CBraMod/blob/e0002aaa87e77a954ce3a11cb1de1ea13e3585e2/finetune_evaluator.py
		with torch.no_grad():
			pred_y = torch.max(prediction, dim=-1)[1]
			truths += ground_truth.clone().cpu().squeeze().numpy().tolist()
			preds += pred_y.clone().cpu().squeeze().numpy().tolist()
			curr_balanced_accuracy = balanced_accuracy_score(ground_truth.clone().cpu().squeeze().numpy(), pred_y.clone().cpu().squeeze().numpy())
			curr_f1 = f1_score(ground_truth.clone().cpu().squeeze().numpy(), pred_y.clone().cpu().squeeze().numpy(), average='weighted')


		shared_features = [encoding for band, encoding in encodings.items()]
		optimizer.zero_grad()
		#task_loss.backward()
		mtl_backward(losses=[task_loss, delta_loss, theta_loss, alpha_loss, beta_loss, gamma_loss], features=shared_features, aggregator=aggregator)
		optimizer.step()
		optimizer.scheduler_step(epoch * len(train_loader) + iteration)
		task_loss_sum += task_loss.item()
		recon_loss = delta_loss.item() + theta_loss.item() + alpha_loss.item() + beta_loss.item() + gamma_loss.item()
		recon_loss_sum += recon_loss
		pbar.set_postfix(task_loss=task_loss.item(), recon_loss=recon_loss, balanced_accuracy=f'{curr_balanced_accuracy*100:.3f}', F1=f'{curr_f1*100:.3f}', lr=optimizer.scheduler.get_last_lr()[0])

	truths = np.array(truths)
	preds = np.array(preds)

	balanced_accuracy = balanced_accuracy_score(truths, preds)
	f1 = f1_score(truths, preds, average='weighted')
	print(f'Mean Acc: {balanced_accuracy} F1 Score: {f1}')
	return task_loss_sum, recon_loss_sum

In [13]:
def validate_one_epoch(model, decoder, task_loss_fn, recon_loss_fn, val_loader):
	task_loss_sum = 0
	recon_loss_sum = 0
	model.eval()
	decoder.eval()

	pbar = tqdm.trange(len(val_loader), desc="Validating")
	data_iterator = iter(val_loader)
	balanced_accuracies = []
	auc_pr = []
	auroc = []

	truths = []
	preds = []
	with torch.no_grad():
		for iteration in pbar:
			data = next(data_iterator)
			relevant_bands = [data['delta'].float().to(device),
								data['theta'].float().to(device),
								data['alpha'].float().to(device),
								data['beta'].float().to(device),
								data['gamma'].float().to(device)]
			inputs = dict(zip(BANDS, relevant_bands))
			encoder_outputs = model(data['graph'], inputs)
			patchified_inputs, encodings, decodings, wavelet_manifold_output, combined_manifold_output = model(data['graph'], inputs)
			wavelet_manifold_output_relevant_patch = dict()
			for band, wavelet_manifold in wavelet_manifold_output.items():
				wavelet_manifold_output_relevant_patch[band] = wavelet_manifold[:, 1, :, :][:, None, :, :]
			prediction = decoder(wavelet_manifold_output_relevant_patch, combined_manifold_output[:, 1, :, :][:, None, :, :]).float()
			ground_truth = data['graph'].y.to(device).long()

			task_loss = task_loss_fn(prediction, ground_truth)

			delta_loss = calculate_band_loss(patchified_inputs['delta'], decodings['delta'], recon_loss_fn)
			theta_loss = calculate_band_loss(patchified_inputs['theta'], decodings['theta'], recon_loss_fn)
			alpha_loss = calculate_band_loss(patchified_inputs['alpha'], decodings['alpha'], recon_loss_fn)
			beta_loss = calculate_band_loss(patchified_inputs['beta'],   decodings['beta'], recon_loss_fn)
			gamma_loss = calculate_band_loss(patchified_inputs['gamma'], decodings['gamma'], recon_loss_fn)

			pred_y = torch.max(prediction, dim=-1)[1]
			truths += ground_truth.clone().cpu().squeeze().numpy().tolist()
			preds += pred_y.clone().cpu().squeeze().numpy().tolist()

			task_loss_sum += task_loss.item()
			recon_loss = delta_loss.item() + theta_loss.item() + alpha_loss.item() + beta_loss.item() + gamma_loss.item()
			recon_loss_sum += recon_loss

			curr_balanced_accuracy = balanced_accuracy_score(ground_truth.clone().cpu().squeeze().numpy(), pred_y.clone().cpu().squeeze().numpy())
			curr_f1 = f1_score(ground_truth.clone().cpu().squeeze().numpy(), pred_y.clone().cpu().squeeze().numpy(), average='weighted')

			pbar.set_postfix(task_loss=task_loss.item(), recon_loss=recon_loss, balanced_accuracy=f'{curr_balanced_accuracy*100:.3f}', F1=f'{curr_f1*100:.3f}', lr=optimizer.scheduler.get_last_lr()[0])
		truths = np.array(truths)
		preds = np.array(preds)

		balanced_accuracy = balanced_accuracy_score(truths, preds)
		f1 = f1_score(truths, preds, average='weighted')

		print(f'Mean Acc: {balanced_accuracy} F1 Score: {f1}')
	return task_loss_sum, recon_loss_sum, balanced_accuracy, f1

In [14]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [15]:
BANDS = ['delta', 'theta', 'alpha', 'beta', 'gamma']
for epoch_idx in range(15):
    training_task_loss, training_recon_loss = train_one_epoch(model, epoch_idx, model_decoder, optimizer, nn.CrossEntropyLoss(label_smoothing=0.1), nn.MSELoss(), finetune_train_loader)
    eval_task_loss, eval_recon_loss, balanced_accuracy, f1 = validate_one_epoch(model, model_decoder, nn.CrossEntropyLoss(label_smoothing=0.1), nn.MSELoss(), finetune_eval_loader)
    print("*" * 50)
    print(f"Epoch: {epoch_idx}")
    print("Epoch: ", epoch_idx, "Total Training Task Loss: ", training_task_loss, "Total Training Recon Loss:", training_recon_loss, "Total Validation Task Loss: ", eval_task_loss, "Total Validation Recon Loss:", eval_recon_loss)
    print("Epoch: ", epoch_idx, "Total Validation Task Loss: ", eval_task_loss, "Total Validation Recon Loss:", eval_recon_loss)
    print("Epoch: ", epoch_idx, "Eval Acc", balanced_accuracy, "F1", f1)

Training: 100%|██████████| 656/656 [03:15<00:00,  3.36it/s, F1=89.728, balanced_accuracy=58.468, lr=1.01e-7, recon_loss=3.64e+4, task_loss=0.668]


Mean Acc: 0.543175993514563 F1 Score: 0.8585926476776788


Validating: 100%|██████████| 230/230 [01:02<00:00,  3.65it/s, F1=68.694, balanced_accuracy=32.184, lr=1.01e-7, recon_loss=7.34e+3, task_loss=1.02] 


Mean Acc: 0.35104617323106285 F1 Score: 0.6960270231737216
**************************************************
Epoch: 0
Epoch:  0 Total Training Task Loss:  490.39687728881836 Total Training Recon Loss: 7059881.337889016 Total Validation Task Loss:  235.436893761158 Total Validation Recon Loss: 3246380.7359154224
Epoch:  0 Total Validation Task Loss:  235.436893761158 Total Validation Recon Loss: 3246380.7359154224
Epoch:  0 Eval Acc 0.35104617323106285 F1 0.6960270231737216


Training: 100%|██████████| 656/656 [03:11<00:00,  3.42it/s, F1=93.546, balanced_accuracy=76.842, lr=5.02e-5, recon_loss=4.4e+3, task_loss=0.581] 


...

Mean Acc: 0.4940050929502133 F1 Score: 0.739457335411311
**************************************************
Epoch: 29
Epoch:  29 Total Training Task Loss:  299.7169514596462 Total Training Recon Loss: 6646331.258494198 Total Validation Task Loss:  235.90111982822418 Total Validation Recon Loss: 3253284.916111946
Epoch:  29 Total Validation Task Loss:  235.90111982822418 Total Validation Recon Loss: 3253284.916111946
Epoch:  29 Eval Acc 0.4940050929502133 F1 0.739457335411311


Training: 100%|██████████| 656/656 [03:09<00:00,  3.46it/s, F1=99.066, balanced_accuracy=98.000, lr=1e-7, recon_loss=3.5e+3, task_loss=0.447]      


Mean Acc: 0.9609104668027132 F1 Score: 0.9897452164285557


Validating: 100%|██████████| 230/230 [01:02<00:00,  3.68it/s, F1=76.542, balanced_accuracy=66.205, lr=1e-7, recon_loss=7.81e+3, task_loss=1]    

Mean Acc: 0.4897586794408278 F1 Score: 0.7359182625909831
**************************************************
Epoch: 30
Epoch:  30 Total Training Task Loss:  299.47681352496147 Total Training Recon Loss: 6639491.313365102 Total Validation Task Loss:  236.00071996450424 Total Validation Recon Loss: 3253213.767627716
Epoch:  30 Total Validation Task Loss:  236.00071996450424 Total Validation Recon Loss: 3253213.767627716
Epoch:  30 Eval Acc 0.4897586794408278 F1 0.7359182625909831
